In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from scipy.interpolate import interp1d
import io
import pickle
import torch
import torch.nn as nn

In [9]:
class IRBeadClassifier(nn.Module):
    def __init__(self, input_length, num_bead_types):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )
        conv_output_size = 128 * (input_length // 8)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_bead_types),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.classifier(x)

# Load bead classes
with open("model/mlb_classes.pkl", "rb") as f:
    mlb = pickle.load(f)

bead_classes = mlb.classes_

# Rebuild and load model
model = IRBeadClassifier(input_length=3600, num_bead_types=23)
model.load_state_dict(torch.load("model/ir_bead_classifier.pth", map_location=torch.device('cpu')))
model.eval()
print("Model and bead classes loaded!")


Model and bead classes loaded!


In [ ]:
def parse_jdx(file_bytes):
    text = file_bytes.decode('utf-8')
    lines = text.splitlines()

    meta = {}
    xy_lines = []
    in_xy = False

    for line in lines:
        line = line.strip()
        if line.startswith('##END'):
            break
        if line.startswith('##XYDATA'):
            in_xy = True
            continue
        if in_xy:
            xy_lines.append(line)
        elif line.startswith('##'):
            parts = line[2:].split('=', 1)
            if len(parts) == 2:
                meta[parts[0].strip().upper()] = parts[1].strip()
    state = meta.get('STATE', '').lower()
    if 'gas' not in state:
        raise ValueError(f"Incompatible sample state: '{meta.get('STATE', 'unknown')}'. This model only supports gas phase IR spectroscopy.")

    yunits = meta.get('YUNITS', '').upper()
    supported_yunits = ['TRANSMITTANCE', 'ABSORBANCE']
    if not any(unit in yunits for unit in supported_yunits):
        raise ValueError(f"Unsupported Y units: '{meta.get('YUNITS', 'unknown')}'. This model only supports TRANSMITTANCE or ABSORBANCE spectra.")

    xunits   = meta.get('XUNITS', '').upper()
    yunits   = meta.get('YUNITS', '').upper()
    xfactor  = float(meta.get('XFACTOR', 1.0))
    yfactor  = float(meta.get('YFACTOR', 1.0))
    deltax   = float(meta.get('DELTAX', 0.0))

    x_vals, y_vals = [], []
    for line in xy_lines:
        if not line:
            continue
        nums = list(map(float, line.split()))
        x_start = nums[0] * xfactor
        ys = [v * yfactor for v in nums[1:]]
        for i, y in enumerate(ys):
            x_vals.append(x_start + i * deltax)
            y_vals.append(y)

    x_arr = np.array(x_vals)
    y_arr = np.array(y_vals)

    # Convert micrometers → 1/cm (wavenumber)
    if 'MICROM' in xunits:
        x_arr = 10000.0 / x_arr
        # After conversion x may be in reverse order — sort it
        sort_idx = np.argsort(x_arr)
        x_arr = x_arr[sort_idx]
        y_arr = y_arr[sort_idx]

    # Convert transmittance → absorbance
    if 'TRANS' in yunits:
        epsilon = 1e-9
        y_arr = -np.log10(np.clip(y_arr, epsilon, None))

    return x_arr, y_arr

def preprocess_spectrum(x_arr, y_arr, target_length=3600):
    f = interp1d(x_arr, y_arr, bounds_error=False, fill_value=0)
    x_new = np.linspace(400, 4000, target_length)
    y_new = f(x_new)
    return y_new

bead_classes = mlb.classes_  # from earlier MultiLabelBinarizer

upload  = widgets.FileUpload(accept='.jdx', multiple=False, description='Upload JDX')
button  = widgets.Button(description='Predict Beads', button_style='primary')
output  = widgets.Output()

def on_predict(b):
    with output:
        clear_output()
        if not upload.value:
            print("Please upload a JDX file first.")
            return

        # Get uploaded file bytes
        file_info = upload.value[0]
        file_bytes = bytes(file_info['content'])

        try:
            print("Parsing JDX file...")
            x_arr, y_arr = parse_jdx(file_bytes)
            print(f"  Parsed {len(x_arr)} data points")
            print(f"  X range: {x_arr.min():.1f} – {x_arr.max():.1f} cm⁻¹")

            print("Preprocessing spectrum...")
            spectrum = preprocess_spectrum(x_arr, y_arr)

            print("Running model...")
            model.eval()
            with torch.no_grad():
                tensor = torch.tensor(spectrum, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                preds  = model(tensor).squeeze().numpy()

            # Threshold at 0.5
            predicted_beads = [bead_classes[i] for i, p in enumerate(preds) if p > 0.5]

            print("\n" + "="*40)
            print("PREDICTED MARTINI BEADS:")
            print("="*40)
            if predicted_beads:
                for bead in predicted_beads:
                    confidence = preds[list(bead_classes).index(bead)]
                    print(f"  {bead:<6} — confidence: {confidence:.2%}")
            else:
                print("  No beads predicted above 0.5 threshold.")

        except Exception as e:
            print(f"Error: {e}")

button.on_click(on_predict)

print("IR Spectrum → Martini Bead Predictor")
print("Upload a .jdx file and click Predict Beads\n")
display(upload, button, output)

IR Spectrum → Martini Bead Predictor
Upload a .jdx file and click Predict Beads



FileUpload(value=(), accept='.jdx', description='Upload JDX')

Button(button_style='primary', description='Predict Beads', style=ButtonStyle())

Output()